In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# from google.colab import drive

# 1. Mount Drive (Essential for accessing your saved data.zip)
# drive.mount('/content/drive')
# DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'

In [2]:
DESTINATION_DIR = '../data'

In [6]:
df = pd.read_pickle(f"{DESTINATION_DIR}/final_merged.pkl")

In [7]:
# 1. Safely extract the RAW, high-dimensional embeddings and labels
embeddings = np.stack(df["embedding"].values)
labels = df['label'].map({0: 'Human', 1: 'Agent'}).values

# 2. Calculate the Homogeneity (Cosine Similarity to Centroid)
results = []

for community in ['Human', 'Agent']:
    # Get all high-dimensional embeddings for this specific group
    community_embeddings = embeddings[labels == community]
    
    # Calculate the centroid (the mathematical "center of gravity" of the community)
    centroid = np.mean(community_embeddings, axis=0).reshape(1, -1)
    
    # Calculate how far every single post is from that center point
    similarities = cosine_similarity(community_embeddings, centroid).flatten()
    
    # Average it out to get the final Homogeneity Score
    avg_similarity = np.mean(similarities)
    
    results.append({
        'Community': community,
        'Homogeneity Score': avg_similarity
    })

df_homogeneity = pd.DataFrame(results)

# 3. Create the Bar Chart for Slide 11
fig = px.bar(
    df_homogeneity,
    x='Community',
    y='Homogeneity Score',
    color='Community',
    color_discrete_map={'Human': '#1f77b4', 'Agent': '#d62728'},
    text_auto='.3f', # Show exact numbers on the bars
    title="Slide 11: Semantic Homogeneity (Rigid Echo Chambers vs Cognitive Diversity)"
)

# 4. Format for Presentation
fig.update_layout(
    template="plotly_white",
    yaxis_title="Avg Cosine Similarity to Centroid (Higher = More Rigid)",
    xaxis_title="",
    showlegend=False,
    height=600,
    width=800
)

# Keep the text clean and visible above the bars
fig.update_traces(textfont_size=18, textangle=0, textposition="outside", cliponaxis=False)

fig.show()

# Export for your slide deck


c:\Users\Rald999\Anaconda3\lib\site-packages\plotly\express\_core.py:1979: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [5]:
fig.write_image("slide_11_semantic_homogeneity.png", scale=2)

ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
